# Download DE Africa CCI Land Cover data for Ghana study area

This notebook downloads the ESA CCI Land Cover product (`cci_landcover`, 300 m resolution, annual 1992–2022) from Digital Earth Africa's public STAC catalog for a study area in Ghana, using `pystac-client` + `odc-stac`.

No Digital Earth Africa Sandbox account or local Open Data Cube is required — this pulls data directly from DE Africa's open S3 archive via STAC.

**Requirements:** see `requirements.txt` in the repo root. Package: `deafrica-tools` is used for plotting with the correct CCI legend.

## 1. Load packages

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import geopandas as gpd
from pystac_client import Client
from odc.stac import stac_load
from deafrica_tools.plotting import plot_lulc

## 2. Define study area

Either supply a bounding box directly, or (recommended) load your own study area boundary file (shapefile / GeoJSON) and derive the bounding box from it.

Replace `study_area_path` below with the path to your AOI file, or edit `bbox` directly if you don't have one yet. The placeholder bbox below covers Ghana's full extent — narrow it down to your actual study area for a faster, smaller download.

In [2]:
# Option A: use your own study area boundary file
study_area_path = None  # e.g. "../data/study_area.geojson"

if study_area_path:
    aoi = gpd.read_file(study_area_path).to_crs("EPSG:4326")
    bbox = list(aoi.total_bounds)  # [min_lon, min_lat, max_lon, max_lat]
else:
    # Option B: manual bbox placeholder covering all of Ghana
    # [min_lon, min_lat, max_lon, max_lat]
    bbox = [-3.5, 4.5, 1.5, 11.5]

print("Bounding box (min_lon, min_lat, max_lon, max_lat):", bbox)

Bounding box (min_lon, min_lat, max_lon, max_lat): [-3.5, 4.5, 1.5, 11.5]


## 3. Set the year(s) of interest

`cci_landcover` is available annually from 1992 to 2022. Set one year, or a list of years to compare land cover change over time.

In [ ]:
years = list(range(1995, 2023))  # 1995–2022 inclusive; CCI annual coverage is 1992–2022

## 4. Search and load data from the DE Africa STAC catalog

In [ ]:
from odc.stac import configure_rio

# DE Africa's S3 buckets are public but require anonymous (unsigned) access —
# without this, rasterio assumes AWS credentials are needed and fails.
configure_rio(
    cloud_defaults=True,
    aws={"aws_unsigned": True},
    AWS_S3_ENDPOINT="s3.af-south-1.amazonaws.com",
)

catalog = Client.open("https://explorer.digitalearth.africa/stac")

datasets = {}

for year in years:
    query = catalog.search(
        collections=["cci_landcover"],
        bbox=bbox,
        datetime=f"{year}-01-01/{year}-12-31",
    )
    items = list(query.items())
    print(f"{year}: found {len(items)} item(s)")

    ds = stac_load(
        items,
        bands=["classification"],
        bbox=bbox,
        resolution=0.0027,  # ~300 m in degrees
    )
    datasets[year] = ds.squeeze()

datasets[years[0]]

## 5. Plot the land cover map(s)

In [ ]:
for year, ds in datasets.items():
    fig, ax = plt.subplots(figsize=(10, 10))
    plot_lulc(ds.classification, product="CCI", legend=True, ax=ax)
    ax.set_title(f"CCI Land Cover — {year}")
    plt.tight_layout()
    plt.show()

## 6. Save data locally (optional)

Export each year's classification layer to GeoTIFF for use in QGIS or elsewhere.

In [ ]:
import os

import rioxarray  # noqa: F401 — registers the .rio accessor

output_dir = "../../data/raw/land_cover/esa"
os.makedirs(output_dir, exist_ok=True)

for year, ds in datasets.items():
    out_path = os.path.join(output_dir, f"cci_landcover_ghana_{year}.tif")
    ds.classification.rio.to_raster(out_path)
    print(f"Saved: {out_path}")